In [17]:
import pandas as pd

df = pd.read_csv(
    "/Users/nguyentien/Documents/DHV/dirty_cafe_sales_CleanPractice.csv",
    dtype=str,
    keep_default_na=False
)
print(df.shape)
print(df.head())
print(df.columns)

(10000, 8)
  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  
Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')


dtype=str: đọc tất cả cột dưới dạng chuỗi để không bị mất thông tin lỗi như ERROR, UNKNOWN.

keep_default_na=False: giữ ô trống là chuỗi rỗng, giúp mình tự kiểm soát missing value.

In [16]:
#Kiểm tra missing values trong từng cột
missing_values = ["", "UNKNOWN", "ERROR"]

for column in df.columns:
    blank_count = (df[column].str.strip() == "").sum()
    unknown_count = (df[column].str.strip().str.upper() == "UNKNOWN").sum()
    error_count = (df[column].str.strip().str.upper() == "ERROR").sum()
    
    
    print(f"Column: {column}")
    print(f"  Blank: {blank_count}")
    print(f"  UNKNOWN: {unknown_count}")
    print(f"  ERROR: {error_count}")
    print()


Column: Transaction ID
  Blank: 0
  UNKNOWN: 0
  ERROR: 0

Column: Item
  Blank: 333
  UNKNOWN: 344
  ERROR: 292

Column: Quantity
  Blank: 138
  UNKNOWN: 171
  ERROR: 170

Column: Price Per Unit
  Blank: 179
  UNKNOWN: 164
  ERROR: 190

Column: Total Spent
  Blank: 173
  UNKNOWN: 165
  ERROR: 164

Column: Payment Method
  Blank: 2579
  UNKNOWN: 293
  ERROR: 306

Column: Location
  Blank: 3265
  UNKNOWN: 338
  ERROR: 358

Column: Transaction Date
  Blank: 159
  UNKNOWN: 159
  ERROR: 142



In [15]:
#Kiểm tra các cột số
for column in ["Quantity", "Price Per Unit", "Total Spent"]:
    numeric_column = pd.to_numeric(df[column], errors="coerce")
    print(f"Column: {column}")
    print("Số dòng chuyển được sang số:", numeric_column.notna().sum())
    print("Min:", numeric_column.min())
    print("Max:", numeric_column.max())
    print("Mean:", numeric_column.mean())
    

Column: Quantity
Số dòng chuyển được sang số: 9521
Min: 1.0
Max: 5.0
Mean: 3.028463396702027
Column: Price Per Unit
Số dòng chuyển được sang số: 9467
Min: 1.0
Max: 5.0
Mean: 2.949984155487483
Column: Total Spent
Số dòng chuyển được sang số: 9498
Min: 1.0
Max: 25.0
Mean: 8.924352495262161


pd.to_numeric(..., errors="coerce"): giá trị nào không chuyển được sang số thì thành NaN.

Các cột số bị lỗi vì có UNKNOWN, ERROR, ô trống.

Quantity hợp lệ nằm từ 1 đến 5.

Price Per Unit hợp lệ nằm trong các giá như 1.0, 1.5, 2.0, 3.0, 4.0, 5.0.

Total Spent phải bằng Quantity * Price Per Unit.

In [14]:
temp = df.copy()

temp["Price Per Unit"] = pd.to_numeric(temp["Price Per Unit"], errors="coerce")

valid = temp[
    (~temp["Item"].isin(["", "UNKNOWN", "ERROR"])) &
    temp["Price Per Unit"].notna()
]

price_map = valid.groupby("Item")["Price Per Unit"].unique()
print(price_map)

Item
Cake        [3.0]
Coffee      [2.0]
Cookie      [1.0]
Juice       [3.0]
Salad       [5.0]
Sandwich    [4.0]
Smoothie    [4.0]
Tea         [1.5]
Name: Price Per Unit, dtype: object


Quy luật này rất quan trọng, vì có thể dùng nó để điền lại Price Per Unit khi biết Item

In [19]:
# Convert datatype

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

df["Price Per Unit"] = pd.to_numeric(
    df["Price Per Unit"],
    errors="coerce"
)

df["Total Spent"] = pd.to_numeric(
    df["Total Spent"],
    errors="coerce"
)

In [13]:
#Tính Total Spent
mask = (
    df["Quantity"].notna() &
    df["Price Per Unit"].notna() &
    df["Total Spent"].isna()
)

df.loc[mask, "Total Spent"] = (
    df.loc[mask, "Quantity"] *
    df.loc[mask, "Price Per Unit"]
)

print(df["Total Spent"].isna().sum())

0


In [12]:
#Recover Quantity
mask = (
    df["Quantity"].isna() &
    df["Price Per Unit"].notna() &
    df["Total Spent"].notna()
)

for idx in df[mask].index:

    q = (
        df.at[idx, "Total Spent"] /
        df.at[idx, "Price Per Unit"]
    )

    # chỉ chấp nhận số nguyên 1-5
    if q == int(q) and 1 <= int(q) <= 5:
        df.at[idx, "Quantity"] = int(q)

print(df["Quantity"].isna().sum())

0


In [11]:
#Recover Price Per Unit
valid_prices = {1.0, 1.5, 2.0, 3.0, 4.0, 5.0}

mask = (
    df["Price Per Unit"].isna() &
    df["Quantity"].notna() &
    df["Total Spent"].notna()
)

for idx in df[mask].index:

    p = (
        df.at[idx, "Total Spent"] /
        df.at[idx, "Quantity"]
    )

    p = round(p, 4)

    if p in valid_prices:
        df.at[idx, "Price Per Unit"] = p

print(df["Price Per Unit"].isna().sum())

0


In [20]:
#Validate Total Spent
check_total = (
    df["Quantity"] * df["Price Per Unit"]
)

wrong_total = df[
    check_total != df["Total Spent"]
]

print("Wrong Total Spent rows:")
print(len(wrong_total))

Wrong Total Spent rows:
1456


In [21]:
#Validate Quantity Range
invalid_quantity = df[
    (df["Quantity"] < 1) |
    (df["Quantity"] > 5)
]

print(len(invalid_quantity))

0


In [22]:
#Validate Price Per Unit
valid_prices = {
    1.0, 1.5, 2.0,
    3.0, 4.0, 5.0
}

invalid_price = df[
    ~df["Price Per Unit"].isin(valid_prices)
]

print(len(invalid_price))

533


In [23]:
#Validate Transaction Date
invalid_date = df[
    df["Transaction Date"].isna()
]

print(len(invalid_date))

0


In [24]:
#Export cleaned data
df.to_csv(
    "cleaned_cafe_sales.csv",
    index=False,
    encoding="utf-8-sig"
)